# 03 — Normalisation, validation et résolution du code postal

Les codes source sont qualifiés avant usage. Une valeur non validable peut être conservée comme source explicite, mais elle n'est jamais utilisée comme référence d'inférence. Les placeholders suspects sont exclus. L'inférence utilise d'abord la localisation exacte normalisée, puis `(ville, pays)`, avant cache/géocodage et overrides ciblés.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from stages_cleaning.config import load_csv_config
from stages_cleaning.geocode import complete_postal_cache, location_key
from stages_cleaning.postal import (
    apply_postal_geocode_cache, apply_postal_overrides, extract_embedded_postal_code,
    finalize_postal_resolution, infer_postal_from_valid_source, normalize_postal_code,
    prepare_postal_resolution, validate_postal_code,
)

INPUT_PATH = ROOT / "data" / "interim" / "stages_efrei_with_country.csv"
OUTPUT_PATH = ROOT / "data" / "interim" / "stages_efrei_with_postal_code.csv"
CACHE_PATH = ROOT / "data" / "interim" / "postal_code_geocoding_cache.csv"
from stages_cleaning.paths import dataset_paths, network_enabled

OVERRIDE_PATH = dataset_paths(ROOT).postal_overrides
# Décision d'équipe : RUN_NETWORK_GEOCODING reste à True par défaut (contrairement à la
# recommandation initiale de l'audit round 3, cf. AUDIT_FIXES.md B-1). Le géocodage réseau
# résout une part significative des pays/CP manquants et le cache rend les runs suivants
# hors ligne. Les risques identifiés (reproductibilité, RGPD, cache empoisonné par une panne
# réseau) sont couverts par les correctifs livrés : geocode.py ne met plus jamais une panne
# réseau en cache comme "not_found" (B-2), et les notebooks ne versionnent plus leurs sorties
# nominatives (R-1). Repasser à False pour un run strictement hors ligne / reproductible.
RUN_NETWORK_GEOCODING = network_enabled()  # STAGES_RUN_NETWORK=0 pour un run hors ligne

df = pd.read_csv(INPUT_PATH, dtype="string", encoding="utf-8-sig", keep_default_na=False).replace("", pd.NA)
overrides = load_csv_config(OVERRIDE_PATH, {"stage_id", "corrected_postal_code", "reason"})
print("Shape entrée :", df.shape)

## 1. Qualification de la source

In [ ]:
df = prepare_postal_resolution(df)
print(df["postal_code_source_status"].value_counts(dropna=False))
print("Placeholders suspects :", int(df["postal_code_source_status"].eq("suspicious_placeholder").sum()))

## 2. Extraction depuis l’adresse

In [ ]:
extracted = pd.Series([extract_embedded_postal_code(address, country) for address, country in zip(df["adresse"], df["pays"])], index=df.index, dtype="string")
mask = df["code_postal_final"].isna() & extracted.notna()
df.loc[mask, "code_postal_final"] = extracted[mask]
df.loc[mask, "postal_code_resolution_method"] = "embedded_in_address"
print("Codes extraits de l'adresse :", int(mask.sum()))

## 3. Inférences internes validées

In [ ]:
df = infer_postal_from_valid_source(df)
print(df["postal_code_resolution_method"].value_counts(dropna=False))
print("Encore sans valeur :", int(df["code_postal_final"].isna().sum()))

## 4. Cache/géocodage contraint par pays

In [ ]:
remaining = df["code_postal_final"].isna()
locations = df.loc[remaining, ["adresse", "ville", "pays"]].drop_duplicates().copy()
locations["location_key"] = location_key(locations, ["adresse", "ville", "pays"])
cache = complete_postal_cache(locations, CACHE_PATH, normalize_postal_code, validate_postal_code, run_network=RUN_NETWORK_GEOCODING)
df["location_key"] = location_key(df, ["adresse", "ville", "pays"])
df = apply_postal_geocode_cache(df, cache)
print("Codes ajoutés depuis cache/géocodage :", int(df["postal_code_resolution_method"].str.endswith("_geocode", na=False).sum()))

## 5. Overrides ciblés, validation et statuts finaux

In [ ]:
df = apply_postal_overrides(df, overrides)
df = finalize_postal_resolution(df)
print(df["postal_code_final_status"].value_counts(dropna=False))
print(df["postal_code_resolution_method"].value_counts(dropna=False))
assert not df.loc[df["postal_code_source_status"].eq("suspicious_placeholder") & df["postal_code_resolution_method"].str.startswith("source", na=False), "code_postal"].notna().any(), "Un placeholder a été accepté comme source"

## 6. Export intermédiaire uniquement

In [ ]:
drop_cols = ["code_postal_final", "_city_key", "_address_key", "location_key", "postal_code_geocode", "geocode_method", "geocode_query"]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print("Export :", OUTPUT_PATH)
print("Shape :", df.shape)